In [10]:
import pandas as pd
import numpy as np

file_path = 'datasets/titanic.csv'
df = pd.read_csv(file_path)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [11]:
def clean_titanic(df):
    df = df.copy()  # never mutate the original
    
    # Drop high-null columns
    df = df.drop(columns=['Cabin'])
    
    # Fill missing values
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    
    # Fix dtypes
    df['Pclass'] = df['Pclass'].astype('category')
    df['Survived'] = df['Survived'].astype('category')
    
    # String cleaning
    df['Sex'] = df['Sex'].str.strip().str.lower()
    
    # Feature engineering
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    return df

df = pd.read_csv(file_path)
df_clean = clean_titanic(df)
print(df_clean.shape)
print(df_clean.isnull().sum())

(891, 13)
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
FamilySize     0
IsAlone        0
dtype: int64


In [12]:
mean_age = df['Age'].mean()
median_age = df['Age'].median()

mean_fare = df['Fare'].mean()
median_fare = df['Fare'].median()

std_age = df['Age'].std()
std_fare = df['Fare'].std()

iqr_age = np.percentile(df['Age'], 75) - np.percentile(df['Age'], 25)
iqr_fare = np.percentile(df['Fare'], 75) - np.percentile(df['Fare'], 25)

print(f"Mean Age: {mean_age:.2f}")
print(f"Median Age: {median_age:.2f}")
print(f"Standard Deviation of Age: {std_age:.2f}")
print(f"Interquartile Range of Age: {iqr_age:.2f}")
print()
print(f"Mean Fare: {mean_fare:.2f}")
print(f"Median Fare: {median_fare:.2f}")
print(f"Interquartile Range of Fare: {iqr_fare:.2f}")
print(f"Standard Deviation of Fare: {std_fare:.2f}")

Mean Age: 29.70
Median Age: 28.00
Standard Deviation of Age: 14.53
Interquartile Range of Age: nan

Mean Fare: 32.20
Median Fare: 14.45
Interquartile Range of Fare: 23.09
Standard Deviation of Fare: 49.69


In [13]:
from scipy import stats

# compare women and men survival stats
survival_stats = stats.ttest_ind(df[df['Sex'] == 'female']['Survived'], df[df['Sex'] == 'male']['Survived'])
print(f"T-test for survival between women and men: t-statistic = {survival_stats.statistic:.2f}, p-value = {survival_stats.pvalue:.4f}")

T-test for survival between women and men: t-statistic = 19.30, p-value = 0.0000


In [14]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['Sex'], df['Survived'])
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"chi2: {chi2:.2f}, p-value: {p:.2e}")

chi2: 260.72, p-value: 1.20e-58
